In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
temp_df = pd.read_csv('IMDB Dataset.csv')

In [3]:
df = temp_df[:10000]

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.shape

(10000, 2)

In [5]:
df['review'][1]

'A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master\'s of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell\'s murals decorating every surface) are terribly well d

In [6]:
df['sentiment'].value_counts()

sentiment
positive    5028
negative    4972
Name: count, dtype: int64

In [7]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(17)

In [9]:
df.drop_duplicates(inplace=True)

In [10]:
df.duplicated().sum()

np.int64(0)

## Pre-processing

In [11]:
import re
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [12]:
# Remove HTML Tags
df['review'] = df['review'].apply(remove_tags)

In [13]:
df['review'] = df['review'].apply(lambda x:x.lower()) # Lowercasing

In [14]:
from nltk.corpus import stopwords

sw_list = stopwords.words('english') 
df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

## Dataset Split

In [15]:
X = df.iloc[:,0:1]
y = df['sentiment']

X.shape , y.shape

((9983, 1), (9983,))

In [16]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(y)

In [17]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_train.shape , X_test.shape

((7986, 1), (1997, 1))

## Using Bag of Words

In [18]:
from sklearn.feature_extraction.text import CountVectorizer

In [19]:
cv = CountVectorizer(max_features=3000)

In [20]:
X_train_bow = cv.fit_transform(X_train['review']).toarray()

X_test_bow = cv.transform(X_test['review']).toarray()

In [21]:
X_train_bow.shape , X_test_bow.shape

((7986, 3000), (1997, 3000))

In [22]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()

gnb.fit(X_train_bow,y_train)

GaussianNB()

In [23]:
from sklearn.metrics import accuracy_score,confusion_matrix

y_pred = gnb.predict(X_test_bow)

accuracy_score(y_test,y_pred)

0.7881822734101152

In [24]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()

rf.fit(X_train_bow,y_train)
y_pred = rf.predict(X_test_bow)

accuracy_score(y_test,y_pred)

0.8407611417125689

## Using N-grams

In [25]:
cv = CountVectorizer(ngram_range=(1,2),max_features=5000)

X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()


In [26]:
rf = RandomForestClassifier()

rf.fit(X_train_bow,y_train)
y_pred = rf.predict(X_test_bow)

accuracy_score(y_test,y_pred)

0.8402603905858789

## Using Tf-IDF

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [29]:
tfidf = TfidfVectorizer(max_features=5000)

In [30]:
X_train_tfidf = tfidf.fit_transform(X_train['review']).toarray()

X_test_tfidf = tfidf.transform(X_test['review'])

In [31]:
rf = RandomForestClassifier()

rf.fit(X_train_tfidf,y_train)
y_pred = rf.predict(X_test_tfidf)

accuracy_score(y_test,y_pred)

0.8377566349524287

## Using Word2Vec

In [40]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess
import gensim

In [39]:
story = []
for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    
    for sent in raw_sent:
        story.append(simple_preprocess(sent))

In [41]:
model = gensim.models.Word2Vec(window=10,min_count=2)

In [42]:
model.build_vocab(story)

In [43]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(5849763, 6186875)

In [ ]:
len(model.wv.index_to_key)  # Total unique words in the corpus

31845

In [46]:
# Average Word2Vec

def document_vector(doc):
    
    doc = [word for word in doc.split() if word in model.wv.index_to_key]  # remove out-of-vocabulary words

    return np.mean(model.wv[doc], axis=0)

In [ ]:
document_vector(df['review'].values[0]).shape

(100,)

In [49]:
from tqdm import tqdm

In [ ]:
# Converting each review to Vector(100-D)
X = []

for doc in tqdm(df['review'].values):
    
    X.append(document_vector(doc))

100%|██████████| 9983/9983 [02:56<00:00, 56.66it/s]


In [51]:
X = np.array(X)

X.shape

(9983, 100)

In [52]:
y = encoder.fit_transform(df['sentiment'])

y.shape

(9983,)

In [53]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_train.shape , X_test.shape

((7986, 100), (1997, 100))

In [54]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)

y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.7776664997496244